# 3-3: MislocScore Analysis

This notebook analyzes batch-normalized mislocalization scores (MislocScores) for VarChAMP classification results.

## Background

The raw AUROC metric from classifiers has two issues:
1. **Counter-intuitive direction**: AUROC ~0.5 = normal, ~1 = mislocalized
2. **Batch variability**: Different batches have different null distribution widths (e.g., Batch 11-12 has tighter distribution than others)

## MislocScore Approaches

We implemented and compared four normalization approaches:

1. **ECDF (Percentile)**: `percentileofscore(ctrl_aurocs, auroc) / 100`
   - Best cross-batch consistency (std of batch means ~0.0008)
   - Interpretation: "This variant exceeds X% of controls"

2. **Effect Size CDF**: `norm.cdf((auroc - ctrl_mean) / ctrl_std)`
   - Good consistency with probability interpretation
   - Interpretation: "P(variant exceeds random control) = score"

3. **Min-Max Scaling**: Linear scaling between batch percentiles
4. **Sigmoid Z-score**: Sigmoid-transformed z-score

## Universal Thresholds
- **Lenient hit**: MislocScore > 0.95
- **Stringent hit**: MislocScore > 0.99

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

# Paths
BASE_DIR = Path('.').resolve().parent.parent
OUTPUT_DIR = BASE_DIR / '3.downstream_analyses' / 'outputs'
MISLOC_COMPARISON_DIR = OUTPUT_DIR / 'misloc_score_comparison'
HIT_CALLS_DIR = OUTPUT_DIR / 'hit_calls'

print(f"Base directory: {BASE_DIR}")
print(f"MislocScore comparison data: {MISLOC_COMPARISON_DIR}")

## 1. Load Comparison Test Results

In [ ]:
# Load the scored classifier data from comparison test
scored_df = pl.read_csv(MISLOC_COMPARISON_DIR / 'all_classifiers_scored.csv')
print(f"Loaded {len(scored_df)} scored classifiers")
print(f"\nBatches: {scored_df['Batch_Short'].unique().to_list()}")
scored_df.head()

In [ ]:
# Load summary statistics
summary_df = pl.read_csv(MISLOC_COMPARISON_DIR / 'summary_statistics.csv')
summary_df

## 2. Cross-Batch Consistency Analysis

The key metric is **cross-batch consistency** - measured as the standard deviation of control means across batches. Lower values indicate better normalization.

In [ ]:
# Plot cross-batch consistency comparison
fig, ax = plt.subplots(figsize=(10, 5))

scores = summary_df['Score'].to_list()
consistency = summary_df['cross_batch_std_of_means'].to_numpy()

colors = ['red' if s == 'AUROC' else 'steelblue' for s in scores]
bars = ax.bar(range(len(scores)), consistency, color=colors)
ax.set_xticks(range(len(scores)))
ax.set_xticklabels(['Raw AUROC', 'ECDF\n(Percentile)', 'Sigmoid', 'Min-Max', 'Effect CDF'], rotation=0)
ax.set_ylabel('Std of Batch Means (lower = better)')
ax.set_title('Cross-Batch Consistency Comparison')

# Add value labels
for bar, val in zip(bars, consistency):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, 
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(MISLOC_COMPARISON_DIR / 'cross_batch_consistency.png', dpi=150)
plt.show()

## 3. Batch 11-12 vs Other Batches

Batch 11-12 has a tighter null distribution than other batches. Let's see how well each approach normalizes this.

In [ ]:
# Compare control distributions: Batch 11-12 vs others
ctrl_df = scored_df.filter(pl.col('Metadata_Control') == True).to_pandas()
ctrl_df['Batch_Group'] = ctrl_df['Batch_Short'].apply(
    lambda x: 'Batch 11-12' if x in ['11', '12'] else 'Other Batches'
)

score_cols = ['AUROC', 'MislocScore_ecdf', 'MislocScore_cdf']
score_labels = ['Raw AUROC', 'ECDF (Percentile)', 'Effect Size CDF']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes, score_cols, score_labels):
    for group in ['Batch 11-12', 'Other Batches']:
        data = ctrl_df[ctrl_df['Batch_Group'] == group][col]
        ax.hist(data, bins=30, alpha=0.5, label=group, density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'Control Distribution: {label}')
    ax.legend()

plt.tight_layout()
plt.savefig(MISLOC_COMPARISON_DIR / 'batch_11_12_comparison.png', dpi=150)
plt.show()

In [ ]:
# Quantitative comparison: control mean and std by batch group
for col, label in zip(score_cols, score_labels):
    print(f"\n{label}:")
    for group in ['Batch 11-12', 'Other Batches']:
        data = ctrl_df[ctrl_df['Batch_Group'] == group][col]
        print(f"  {group}: mean={data.mean():.4f}, std={data.std():.4f}")

## 4. Hit Calling Comparison

In [ ]:
# Load combined hit calls
hit_calls_path = HIT_CALLS_DIR / 'combined_hit_calls.csv'
if hit_calls_path.exists():
    hits_df = pl.read_csv(hit_calls_path)
    print(f"Loaded {len(hits_df)} variants from hit calls")
    hits_df.head()
else:
    print("Hit calls not found. Run generate_hit_calls.py first.")

In [ ]:
# Compare hit rates between original AUROC method and MislocScore method
if hit_calls_path.exists():
    hit_comparison = {
        'Method': ['AUROC 95th', 'AUROC 99th', 'MislocScore Mean 95', 'MislocScore Mean 99'],
        'Hits': [
            hits_df.filter(pl.col('hit_multichannel_95')).height,
            hits_df.filter(pl.col('hit_multichannel_99')).height,
            hits_df.filter(pl.col('hit_misloc_mean_95')).height,
            hits_df.filter(pl.col('hit_misloc_mean_99')).height,
        ]
    }
    hit_comparison_df = pd.DataFrame(hit_comparison)
    hit_comparison_df['Percentage'] = hit_comparison_df['Hits'] / len(hits_df) * 100
    print(hit_comparison_df.to_string(index=False))

## 5. MislocScore Distribution by Batch

In [ ]:
if hit_calls_path.exists():
    # Get batch short name
    hits_df = hits_df.with_columns(
        pl.col('Batch').str.split('_').list.last().alias('Batch_Short')
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Box plot of MislocScore by batch
    hits_pd = hits_df.to_pandas()
    batches = sorted(hits_pd['Batch_Short'].unique())
    data_by_batch = [hits_pd[hits_pd['Batch_Short'] == b]['MislocScore_ecdf_mean'] for b in batches]
    
    bp = axes[0].boxplot(data_by_batch, labels=batches, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.5)
    axes[0].axhline(0.95, color='green', linestyle='--', alpha=0.7, label='0.95 threshold')
    axes[0].axhline(0.99, color='red', linestyle='--', alpha=0.7, label='0.99 threshold')
    axes[0].set_xlabel('Batch')
    axes[0].set_ylabel('MislocScore (ECDF)')
    axes[0].set_title('MislocScore Distribution by Batch')
    axes[0].legend()
    
    # Right: Scatter plot AUROC vs MislocScore
    axes[1].scatter(hits_pd['AUROC_Mean_Channel'], hits_pd['MislocScore_ecdf_mean'], 
                    alpha=0.5, s=20)
    axes[1].axhline(0.95, color='green', linestyle='--', alpha=0.7)
    axes[1].axhline(0.99, color='red', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('AUROC Mean Channel')
    axes[1].set_ylabel('MislocScore (ECDF)')
    axes[1].set_title('AUROC vs MislocScore')
    
    plt.tight_layout()
    plt.savefig(MISLOC_COMPARISON_DIR / 'misloc_score_by_batch.png', dpi=150)
    plt.show()

## 6. Top Hits Analysis

In [ ]:
if hit_calls_path.exists():
    # Top 20 variants by MislocScore
    top_hits = hits_df.sort('MislocScore_ecdf_mean', descending=True).head(20)
    print("Top 20 variants by MislocScore:")
    print(top_hits.select([
        'Batch', 'allele_0', 'allele_1', 
        'MislocScore_ecdf_mean', 'AUROC_Mean_Channel',
        'hit_misloc_mean_95', 'hit_misloc_mean_99'
    ]))

In [ ]:
if hit_calls_path.exists():
    # Variants that are hits by MislocScore but not by AUROC (or vice versa)
    discordant_misloc_only = hits_df.filter(
        (pl.col('hit_misloc_mean_95') == True) & (pl.col('hit_multichannel_95') == False)
    )
    discordant_auroc_only = hits_df.filter(
        (pl.col('hit_misloc_mean_95') == False) & (pl.col('hit_multichannel_95') == True)
    )
    
    print(f"\nDiscordant hits (95th threshold):")
    print(f"  MislocScore only: {len(discordant_misloc_only)}")
    print(f"  AUROC only: {len(discordant_auroc_only)}")
    
    if len(discordant_auroc_only) > 0:
        print(f"\nVariants called hits by AUROC but not MislocScore (batch effect?):")
        print(discordant_auroc_only.select([
            'Batch', 'allele_0', 'allele_1', 
            'MislocScore_ecdf_mean', 'AUROC_Mean_Channel'
        ]).head(10))

## 7. Summary and Recommendations

Based on the analysis:

1. **ECDF (Percentile)** has the best cross-batch consistency and is recommended as the primary MislocScore method
2. **Effect Size CDF** is a good alternative with probability interpretation
3. Universal thresholds (0.95, 0.99) can be applied across batches
4. The MislocScore effectively normalizes the different null distribution widths between Batch 11-12 and other batches

In [ ]:
print("Analysis complete!")
print(f"\nOutput files saved to: {MISLOC_COMPARISON_DIR}")